In [90]:
import numpy as np
from bingo . evolutionary_algorithms . age_fitness import AgeFitnessEA # evolutionary algorithm
from bingo . evolutionary_optimizers . island import Island 
from bingo.symbolic_regression.symbolic_regressor import SymbolicRegressor
from bingo.symbolic_regression import ComponentGenerator , AGraphGenerator #generadores
from bingo.symbolic_regression import AGraphCrossover, AGraphMutation    #Cruza y mutación
from bingo.symbolic_regression.agraph.agraph import AGraph
from bingo.symbolic_regression.explicit_regression import ExplicitRegression , ExplicitTrainingData
from bingo.evaluation.fitness_function import VectorBasedFunction
from bingo.evaluation.evaluation import Evaluation
from bingo.local_optimizers.scipy_optimizer import ScipyOptimizer
import matplotlib.pyplot as plt
import math
import sympy as sp
#fitness function
from bingo.local_optimizers.local_opt_fitness import LocalOptFitnessFunction # CLO


In [ ]:
"""
CODIGO OFICIAL DE BINGO

import numpy as np # for generating data
from bingo . evolutionary_algorithms . age_fitness import AgeFitnessEA # evolutionary algorithm
from bingo . evolutionary_optimizers . island import Island # evolutionary optimizer
# components for evaluation
from bingo . evaluation . evaluation import Evaluation
from bingo . symbolic_regression import ExplicitRegression , ExplicitTrainingData 
#fitness function
from bingo . local_optimizers . continuous_local_opt import ContinuousLocalOptimization # CLO
# generators
from bingo . symbolic_regression import ComponentGenerator , AGraphGenerator
# crossover and mutation
from bingo . symbolic_regression import AGraphCrossover, AGraphMutation

# hyperparams
POP_SIZE = 100
STACK_SIZE = 20

# make some data
x = np . linspace ( -10 , 10 , 100) . reshape ([ -1 , 1])
y = x **2 + 3.5* x **3
training_data = ExplicitTrainingData (x , y )

# setup generator to generate equations using variables , constants , + , -, and *
component_generator = ComponentGenerator ( x . shape [1])
component_generator . add_operator (" + ")
component_generator . add_operator (" -")
component_generator . add_operator (" * ")

# agraphs of equations will have STACK_SIZE (20) commands and will use algebraic simplification
agraph_generator = AGraphGenerator ( STACK_SIZE , component_generator , use_simplification = True )

# crossover and mutation
crossover = AGraphCrossover ()
mutation = AGraphMutation ( component_generator )

# defining evaluation
fitness = ExplicitRegression ( training_data =
training_data , metric =" mse ")
local_opt_fitness = ContinuousLocalOptimization (
fitness , algorithm =" lm ")
evaluator = Evaluation ( local_opt_fitness )

# setup evolutionary algorithm
ea = AgeFitnessEA ( evaluator , agraph_generator ,
crossover , mutation , 0.4 , 0.4 , POP_SIZE )

# setup evolutionary optimizer
island = Island ( ea , agraph_generator , POP_SIZE )

# run !
opt_result = island . evolve_until_convergence (
max_generations =500 , fitness_threshold =1.0 e -4)

# print best individual from last generation
print ( opt_result . get_best_individual () )
"""

In [89]:
P = 150
Complejidad = 10

## Primera Ecuación (para agarrar confianza)

$\frac{d y}{dx} - y = 0 \ , \quad y(0) = 1$

$y(x) = e^{x}$

## 1) Iniciar Población

In [96]:
component_generator = ComponentGenerator(1) #el 1 es la dimensión de X
component_generator.add_operator ("+")
component_generator.add_operator ("-")
component_generator.add_operator ("*")
component_generator.add_operator ("/")
component_generator.add_operator ("pow")
component_generator.add_operator ("exp")
component_generator.add_operator ("sin")
component_generator.add_operator ("cos")

agraph_generator = AGraphGenerator(Complejidad ,component_generator , use_simplification = True)

In [156]:
#CADA VEZ QUE SE COMPILA ESTE BLOQUE SE HACE OTRA POBLACIÓN, CUIDADO
poblacion = []
for i in range(P):
    cromosoma = agraph_generator()
    poblacion.append(cromosoma)

print(poblacion[0].get_formatted_string("console"), "\n")
print(poblacion[1].get_formatted_string("console"), "\n")
print(poblacion[2].get_formatted_string("console"), "\n")
print(poblacion[3].get_formatted_string("console"), "\n")
print(poblacion[4].get_formatted_string("console"), "\n")
print(len(poblacion))

1 

1 + exp(1) 

1.000000 

(1.000000)((X_0)^(-1)) 

1.000000 

150


In [98]:
print(poblacion[0].get_formatted_string("console"), "\n")

((X_0)^(-1))((cos(cos(X_0)) - (X_0))^((-1)(cos(cos(X_0)) - (X_0)))) - (X_0) 



### Al final me di cuenta que toda la sección de iniciar población no sirve de nada para el AG, pero me ayudó mucho a entender que estaba pasando, asi que lo voy a dejar tal cual

## 2) Evaluación de individuos

In [148]:
def Make_Xi(seed: int, n_points: int, min = float, max = float):
    rng = np.random.default_rng(seed)
    nosequenombreponerperoesunarray = np.sort(rng.uniform(min, max, size=n_points))
    return nosequenombreponerperoesunarray.reshape(-1, 1)
Xi = Make_Xi(67, 100, 0.01, 0.99)
print(Xi.shape)

(100, 1)


In [113]:
class EDO1(VectorBasedFunction):
    def __init__(self, x, lambdA ):
        super().__init__(metric="mse")
        self.x = x
        self.lambdA = lambdA


    def evaluate_fitness_vector(self, individual):
        y, dy_dx = individual.evaluate_equation_with_x_gradient_at(self.x)

        y = y[:, 0]
        dy_dx = dy_dx[:, 0]


        #DEBE CUMPLIR LA EDO
        residual1 = (dy_dx - y)**2


        #DEBE CUMPLIR LA CONDICIÓN INICIAL  y(0) = 1
        ConIni = individual.evaluate_equation_at(np.array([[0.0]]))[0, 0]

        residual2 = (ConIni - 1.0)**2

        RESIDUALULTRAMASTERpro = residual1 + self.lambdA * residual2
        return RESIDUALULTRAMASTERpro

In [149]:
fitness = EDO1(Xi,1) # Función objetivo
optimizer = ScipyOptimizer(fitness, method="BFGS")
local_opt_fitness = LocalOptFitnessFunction(fitness, optimizer) #Evaluador con optimización local
evaluator = Evaluation(local_opt_fitness)

## Crossover y mutación

In [150]:
crossover = AGraphCrossover()
mutation = AGraphMutation(component_generator)

## Algoritmo

In [151]:
ea = AgeFitnessEA (evaluator , agraph_generator ,crossover , mutation , 0.5 , 0.5 , P)
island = Island(ea , agraph_generator , P)

In [155]:
opt_result = island.evolve_until_convergence(max_generations = 500 , fitness_threshold = 1e-10)

print(island.get_best_individual())

exp(X_0)


In [ ]:
class EDO2(VectorBasedFunction):
    def __init__(self, x, lambdA ):
        super().__init__(metric="mse")
        self.x = x
        self.lambdA = lambdA


    def evaluate_fitness_vector(self, individual):
        c = 5e-5
        
        u = individual.evaluate_equation_at(self.x)

        u = u[:, 0]
        dy_dx = dy_dx[:, 0]


        #DEBE CUMPLIR LA EDO
        residual1 = (du4_dx4 - c)**2


        #DEBE CUMPLIR LA CONDICIÓN INICIAL  y(0) = 1
        ConIni = individual.evaluate_equation_at()

        residual2 = (ConIni - 1.0)**2

        RESIDUALULTRAMASTERpro = residual1 + self.lambdA * residual2
        return RESIDUALULTRAMASTERpro

In [ ]:
class EDO3(VectorBasedFunction):
    def __init__(self, x, lambdA ):
        super().__init__(metric="mse")
        self.x = x
        self.lambdA = lambdA


    def evaluate_fitness_vector(self, individual):
        c = 5e-5
        
        u = individual.evaluate_equation_at(self.x)

        u = u[:, 0]
        dy_dx = dy_dx[:, 0]


        #DEBE CUMPLIR LA EDO
        residual1 = (du4_dx4 - c)**2


        #DEBE CUMPLIR LA CONDICIÓN INICIAL  y(0) = 1
        ConIni = individual.evaluate_equation_at()

        residual2 = (ConIni - 1.0)**2

        RESIDUALULTRAMASTERpro = residual1 + self.lambdA * residual2
        return RESIDUALULTRAMASTERpro